In [1]:
!pip install transformers datasets seqeval -q
print("Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Done


In [16]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score
)

print("All imports done")
print("GPU:", torch.cuda.is_available())

All imports done
GPU: True


In [17]:
dataset = load_dataset("wikiann", "en")
print(dataset)
print("\nSample:")
print(dataset["train"][0])

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

Sample:
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


In [18]:
label_list = dataset["train"]\
    .features["ner_tags"].feature.names

id2label = {
    i: label
    for i, label in enumerate(label_list)
}
label2id = {
    label: i
    for i, label in enumerate(label_list)
}

print("Labels:", label_list)
print("Total:", len(label_list))

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Total: 7


In [19]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)
print("Tokenizer loaded")

Tokenizer loaded


In [8]:
# Already done in Cell 4
print("Label mapping ready")
print("id2label:", id2label)

Label mapping ready
id2label: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}


In [20]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )
    all_labels = []
    for i, labels in enumerate(
            examples["ner_tags"]):
        word_ids = tokenized_inputs\
            .word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(
                    labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Alignment function defined")

Alignment function defined


In [21]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)
print("Tokenization done")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenization done


In [22]:
model = AutoModelForTokenClassification\
    .from_pretrained(
        "distilbert-base-uncased",
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id
    )
print("Model loaded")
print("Num labels:", len(label_list))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded
Num labels: 7


In [23]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)
print("Data collator ready")

Data collator ready


In [24]:
def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(
        predictions, axis=2)
    true_predictions = [
        [label_list[p]
         for p, l in zip(prediction, label)
         if l != -100]
        for prediction, label
        in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l]
         for p, l in zip(prediction, label)
         if l != -100]
        for prediction, label
        in zip(predictions, labels)
    ]
    return {
        "precision": precision_score(
            true_labels, true_predictions),
        "recall": recall_score(
            true_labels, true_predictions),
        "f1": f1_score(
            true_labels, true_predictions)
    }

print("compute_metrics defined")

compute_metrics defined


In [25]:
training_args = TrainingArguments(
    output_dir="./pos_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)
print("Training args defined")

Training args defined


In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
print("Trainer ready")

Trainer ready


In [29]:
print("Training started...")
trainer.train()

Training started...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.289222,0.254468,0.790032,0.817165,0.803369
2,0.205514,0.241395,0.801809,0.828778,0.815070
3,0.160300,0.249193,0.810764,0.835222,0.822811


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3750, training_loss=0.24594867884318034, metrics={'train_runtime': 312.3584, 'train_samples_per_second': 192.087, 'train_steps_per_second': 12.005, 'total_flos': 1959973678080000.0, 'train_loss': 0.24594867884318034, 'epoch': 3.0})

In [30]:
results = trainer.evaluate(
    tokenized_dataset["test"])

print("\n===== RESULTS =====")
for key, val in results.items():
    print(f"{key}: {round(val, 4)}")


===== RESULTS =====
eval_loss: 0.2401
eval_precision: 0.8014
eval_recall: 0.8271
eval_f1: 0.8141
eval_runtime: 23.1935
eval_samples_per_second: 431.156
eval_steps_per_second: 26.947
epoch: 3.0


In [31]:
predictions, labels, _ = trainer.predict(
    tokenized_dataset["test"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p]
     for p, l in zip(prediction, label)
     if l != -100]
    for prediction, label
    in zip(predictions, labels)
]
true_labels = [
    [label_list[l]
     for p, l in zip(prediction, label)
     if l != -100]
    for prediction, label
    in zip(predictions, labels)
]

print("Classification Report:")
print(classification_report(
    true_labels, true_predictions))

Classification Report:
              precision    recall  f1-score   support

         LOC       0.82      0.84      0.83      4641
         ORG       0.72      0.74      0.73      4745
         PER       0.87      0.90      0.89      4518

   micro avg       0.80      0.83      0.81     13904
   macro avg       0.80      0.83      0.82     13904
weighted avg       0.80      0.83      0.81     13904



In [32]:
nlp_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

sentences = [
    "John works at Google in California",
    "Apple is buying a startup in London",
    "Barack Obama was born in Hawaii"
]

for sentence in sentences:
    print(f"\nInput: {sentence}")
    result = nlp_pipeline(sentence)
    print("Tags:")
    for r in result:
        print(f"  {r['word']:15} → "
              f"{r['entity_group']}")


Input: John works at Google in California
Tags:
  john works      → PER
  google          → ORG
  california      → LOC

Input: Apple is buying a startup in London
Tags:
  apple is        → ORG
  london          → LOC

Input: Barack Obama was born in Hawaii
Tags:
  barack obama    → PER
  hawaii          → LOC


In [33]:
print("""
========================================
ASSIGNMENT 5 - TOKEN CLASSIFICATION
========================================
Model   : distilbert-base-uncased
Dataset : WikiANN English
Task    : NER / POS Tagging
Epochs  : 3
LR      : 2e-5

Labels:
- O     : Outside
- B-PER : Person begin
- I-PER : Person inside
- B-ORG : Organization begin
- I-ORG : Organization inside
- B-LOC : Location begin
- I-LOC : Location inside

Key Learning:
Token classification assigns a label
to each individual token in a sentence
rather than the whole sentence.
========================================
""")


ASSIGNMENT 5 - TOKEN CLASSIFICATION
Model   : distilbert-base-uncased
Dataset : WikiANN English
Task    : NER / POS Tagging
Epochs  : 3
LR      : 2e-5

Labels:
- O     : Outside
- B-PER : Person begin
- I-PER : Person inside
- B-ORG : Organization begin
- I-ORG : Organization inside
- B-LOC : Location begin
- I-LOC : Location inside

Key Learning:
Token classification assigns a label
to each individual token in a sentence
rather than the whole sentence.

